#### This notebook is used to create the ground truth data for training LLM.

Problem:
    Currently when we are trying to get justifications from GPT4. It does not give good explanations. Sometimes it says that I dont have "sufficient knowledge to answer this question."

Solution:
    Leverage step by step, task decomposition approach to fetch factual data from web, augment the prompt context for generating justification and get the answer.

Steps tried below:
1. Using Agents approach
   1. Load the data
   2. Create a agent with fixed input and output format that would be used to fetch the data from web.
   3. Create prompt template with data from step 1 and 2.
   4. Output to create evidences and justifications.
   5. Test the agent with single row.
   6. Repeat for all data points
2. Using ProgramFC
   1. Approach given here: https://github.com/teacherpeterpan/ProgramFC/tree/main, Paper: https://arxiv.org/abs/2305.12744
   2. 


#### Approach 1: Agentic

In [1]:

import pandas as pd
import numpy as np

# Read in the data
df = pd.read_csv('../data/assigned-labels-205.csv')


In [2]:
df.describe()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels
count,205,205,205,200,205,205,197,205,205,205,205
unique,204,198,76,196,204,205,196,205,183,183,3
top,swst4yk-ow8,High Return Stocks - Top-Performing Stocks fro...,5paisa,UPCOMING IPO SEPTEMBER 2023 IN INDIA\nLATEST I...,Hello everyone how are you all and what's goi...,"I cannot guarantee you that, but I can assure ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAoNDQ0IDQ...,https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,No claims made,No claims to analyse,True
freq,2,5,60,2,2,1,2,1,23,23,114


In [4]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels
0,dZ7xeVCYC5M,How I Would Invest $1000 If I Were In My 20s,The Game w/ Alex Hormozi,My new book $100M Leads is now LIVE. Grab your...,i [ __ ] guarantee you that you will be makin...,"I cannot guarantee you that, but I can assure ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgKDQoICg...,https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,The influencer claims that investing in self-e...,The influencer's claim that investing in self-...,True
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...,Neutral
2,JnIYdowe9KE,Swing Trading Profit Double - Best Strategy,Stock Learners,In This Video I Have Share One Of My Trade Log...,तो स्टॉक मार्केट में कैसे आप स्विंग ट्रेडिंग क...,So how can you make a good return by swing tra...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBBAQDxAQEB...,https://www.youtube.com/watch?v=JnIYdowe9KE\n,The financial influencer claims that by swing ...,The influencer's claim that one can make signi...,True
3,GiiHU87xuGY,Top 3 positive stocks | Stocks for 23-Oct-2023...,PM Stock Academy,NaN,"वयलकम दोस्तो, बीयम स्टॉक अकडमी के एक नए वीडियो...","""Hello friends, welcome to the video of the Ve...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=GiiHU87xuGY\n,The financial influencer suggests three stocks...,The claims made by the influencer are based on...,False
4,gsXgM6WLJsg,"Turning $100 Into $1,000 Trading Stocks | Ep.1",Jenny Hoyos,get up to 10 FREE stocks (deposit at least $10...,this is a penny and last week i tried turning...,"""This is a penny, and last week I attempted to...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBg0KCQkJCQ...,https://www.youtube.com/watch?v=gsXgM6WLJsg\n,The financial influencer claims that he is att...,"The claim of turning 100 into 1,000 through st...",True


"The influencer's claim that investing in self-education and skills can increase earning capacity is generally true. Numerous studies have shown that individuals with higher levels of education and skills tend to earn more over their lifetimes. However, the claim that this can lead to higher income than traditional investments like the S&P 500 or real estate is more subjective and depends on various factors, including the individual's ability to apply the knowledge and skills acquired, the demand for those skills in the marketplace, and the individual's personal financial situation. The claim that being actively involved in income-generating activities can lead to higher income is also generally true, as active involvement often leads to better decision-making and higher returns. However, this requires time, effort, and expertise, which not everyone may have. The advice to learn from multiple sources is sound, as it can lead to a more well-rounded understanding of a subject. The claim 

In [17]:
df['Justification'][0]

"The influencer's claim that investing in self-education and skills can increase earning capacity is generally true. Numerous studies have shown that individuals with higher levels of education and skills tend to earn more over their lifetimes. However, the claim that this can lead to higher income than traditional investments like the S&P 500 or real estate is more subjective and depends on various factors, including the individual's ability to apply the knowledge and skills acquired, the demand for those skills in the marketplace, and the individual's personal financial situation. The claim that being actively involved in income-generating activities can lead to higher income is also generally true, as active involvement often leads to better decision-making and higher returns. However, this requires time, effort, and expertise, which not everyone may have. The advice to learn from multiple sources is sound, as it can lead to a more well-rounded understanding of a subject. The claim 

In [ ]:
# currenlty used df.apply function to ALL dataset at same time, without any regard for accuracy

class JSONFormatter(BaseModel):
   SummarizedClaims : str = Field(description="summary of claims made by the finfluencer")
   Justification : str = Field(description="justification provided by gpt-4 for the claims")

parser = PydanticOutputParser(pydantic_object=JSONFormatter)

def extract_claims_and_justification(sample):
    transcript = sample["eng_transcript"]
    messages = [
        SystemMessage(content="You are a Financial Analyst. You are provided with the youtube transcript of a fincancial influencer. \
                    Your task is to identify and extract the Claims/Tips/Tricks made by the Financial Influencer. \
                    Summarize the extracted claims into a small paragraph not exceeding 150 words. \
                    Extract all false claims and provide counter thesis or Justification for false claims. \
                    Please separate out the Summarized Claims and Justification from the response. \
                    Provide the response in JSON format containing keys for SummarizedClaims and Justification"),
        HumanMessage(content=transcript)
    ]
    response = chat(messages)
    formatted_output = parser.parse(response.content)
    return [formatted_output.SummarizedClaims, formatted_output.Justification]